# Delivery Optimization Problem

## Objective
- Model and solve a realistic Delivery Optiomization Problem with Time Windows (VRPTW) for urban last-mile delivery optimization by leveraging real-world data from multiple Kaggle datasets.
- Compare solution quality, runtime, scalability, and feasibility of each exact method and heuristic method.

In [20]:
# ===== IMPORTS =====
from __future__ import annotations
import math
import time
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd
import pulp
import warnings
warnings.filterwarnings("ignore")

@dataclass
class SolutionResult:
    method: str
    route: List[int]
    total_minutes: float
    runtime_seconds: float

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


### DATA LOADING AND PREPROCESSING

In [21]:
def load_and_prepare_data(csv_path: str, n_orders: int, seed: int) -> Tuple[Tuple[float, float], pd.DataFrame]:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    for col in ["Traffic", "Weather", "Vehicle", "Area"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    required_cols = [
        "Store_Latitude",
        "Store_Longitude",
        "Drop_Latitude",
        "Drop_Longitude",
        "Traffic",
        "Weather",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df.dropna(subset=required_cols).copy()
    work = work[~work["Traffic"].str.lower().eq("nan")]
    work = work[~work["Weather"].str.lower().eq("nan")]

    store_group = (
        work.groupby(["Store_Latitude", "Store_Longitude"]).size().reset_index(name="count")
    )
    busiest = store_group.sort_values("count", ascending=False).iloc[0]
    depot = (float(busiest["Store_Latitude"]), float(busiest["Store_Longitude"]))

    subset = work[
        (work["Store_Latitude"] == depot[0]) & (work["Store_Longitude"] == depot[1])
    ].copy()

    if len(subset) < n_orders:
        n_orders = len(subset)

    subset = subset.sample(n=n_orders, random_state=seed).reset_index(drop=True)
    return depot, subset

### HAVERSINE & COST MATRIX

In [22]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return r * c


def build_cost_matrix(
    depot: Tuple[float, float],
    orders: pd.DataFrame,
    base_speed_kmph: float = 25.0,
    service_minutes: float = 3.0,
) -> np.ndarray:
    traffic_multiplier = {
        "Low": 1.0,
        "Medium": 1.2,
        "High": 1.5,
        "Jam": 2.0,
    }
    weather_multiplier = {
        "Sunny": 1.0,
        "Cloudy": 1.05,
        "Windy": 1.12,
        "Fog": 1.2,
        "Stormy": 1.3,
        "Sandstorms": 1.4,
    }

    coords = [depot] + list(zip(orders["Drop_Latitude"], orders["Drop_Longitude"]))
    n = len(coords)
    cost = np.zeros((n, n), dtype=float)

    node_multiplier = [1.0]
    for _, row in orders.iterrows():
        t = traffic_multiplier.get(str(row["Traffic"]), 1.2)
        w = weather_multiplier.get(str(row["Weather"]), 1.1)
        node_multiplier.append(t * w)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            d_km = haversine_km(coords[i][0], coords[i][1], coords[j][0], coords[j][1])
            travel_minutes = (d_km / base_speed_kmph) * 60.0
            visit_penalty = service_minutes if j != 0 else 0.0
            cost[i, j] = travel_minutes * node_multiplier[j] + visit_penalty
    return cost


def route_cost(route: List[int], C: np.ndarray) -> float:
    if not route:
        return float("inf")
    total = C[0][route[0]]
    total += sum(C[route[k]][route[k + 1]] for k in range(len(route) - 1))
    total += C[route[-1]][0]
    return total


### ILP (Exact Method)

In [23]:
def solve_exact_tsp_mtz(cost: np.ndarray, time_limit: int = 120) -> SolutionResult:
    start = time.perf_counter()
    n = cost.shape[0]
    nodes = list(range(n))

    model = pulp.LpProblem("DeliveryTSP", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", (nodes, nodes), cat="Binary")
    u = pulp.LpVariable.dicts("u", nodes, lowBound=0, upBound=n - 1, cat="Continuous")

    model += pulp.lpSum(cost[i, j] * x[i][j] for i in nodes for j in nodes if i != j)

    for i in nodes:
        model += pulp.lpSum(x[i][j] for j in nodes if j != i) == 1
        model += pulp.lpSum(x[j][i] for j in nodes if j != i) == 1

    for i in nodes:
        model += x[i][i] == 0

    for i in range(1, n):
        model += u[i] >= 1
        model += u[i] <= n - 1

    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                model += u[i] - u[j] + n * x[i][j] <= n - 1

    solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit)
    model.solve(solver)

    succ = {}
    for i in nodes:
        for j in nodes:
            if i != j and pulp.value(x[i][j]) is not None and pulp.value(x[i][j]) > 0.5:
                succ[i] = j

    route = [0]
    visited = {0}
    cur = 0
    while True:
        nxt = succ.get(cur)
        if nxt is None:
            break
        route.append(nxt)
        if nxt == 0:
            break
        if nxt in visited:
            break
        visited.add(nxt)
        cur = nxt

    total = float(pulp.value(model.objective)) if pulp.value(model.objective) is not None else float("inf")
    runtime = time.perf_counter() - start
    return SolutionResult("Exact-MTZ", route, total, runtime)


### HELD-KARP DYNAMIC PROGRAMMING (Exact Method)

In [24]:
def solve_held_karp(C: np.ndarray) -> dict:
    n_total = C.shape[0]
    n = n_total - 1

    if n == 0:
        return {
            "solver": "Held-Karp DP (Exact)", "route": [], "cost": 0.0,
            "runtime_s": 0.0, "feasible": True, "status": "Trivial (0 nodes)",
        }

    INF  = float("inf")
    FULL = (1 << n) - 1

    dp     = np.full((1 << n, n), INF,  dtype=np.float64)
    parent = np.full((1 << n, n), -1,   dtype=np.int32)

    t0 = time.perf_counter()

    for i in range(n):
        dp[1 << i][i] = C[0, i + 1]

    for mask in range(1, 1 << n):
        for i in range(n):
            if not (mask & (1 << i)):
                continue
            cur_cost = dp[mask][i]
            if cur_cost == INF:
                continue
            for j in range(n):
                if mask & (1 << j):      # already visited
                    continue
                new_mask = mask | (1 << j)
                new_cost = cur_cost + C[i + 1, j + 1]
                if new_cost < dp[new_mask][j]:
                    dp[new_mask][j]     = new_cost
                    parent[new_mask][j] = i

    best_cost  = INF
    last_node  = -1
    for i in range(n):
        if dp[FULL][i] == INF:
            continue
        total = dp[FULL][i] + C[i + 1, 0]
        if total < best_cost:
            best_cost = total
            last_node = i

    route_idx = []
    mask, node = FULL, last_node
    while node != -1:
        route_idx.append(node)
        prev = parent[mask][node]
        mask = mask ^ (1 << node)
        node = prev
    route_idx.reverse()

    route   = [idx + 1 for idx in route_idx]
    elapsed = time.perf_counter() - t0

    return {
        "solver":     "Held-Karp DP (Exact)",
        "route":      route,
        "cost":       best_cost,
        "runtime_s":  elapsed,
        "feasible":   len(route) == n,
        "status":     "Globally optimal",
    }

### LOCAL SEARCH (Heuristic Method)

In [25]:

def route_cost(route: List[int], cost: np.ndarray) -> float:
    return float(sum(cost[route[i], route[i + 1]] for i in range(len(route) - 1)))


def nearest_neighbor_route(cost: np.ndarray) -> List[int]:
    n = cost.shape[0]
    unvisited = set(range(1, n))
    route = [0]
    current = 0
    while unvisited:
        nxt = min(unvisited, key=lambda j: cost[current, j])
        route.append(nxt)
        unvisited.remove(nxt)
        current = nxt
    route.append(0)
    return route


def two_opt(route: List[int], cost: np.ndarray) -> List[int]:
    best = route[:]
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):
                if j - i == 1:
                    continue
                candidate = best[:i] + best[i:j][::-1] + best[j:]
                if route_cost(candidate, cost) + 1e-9 < route_cost(best, cost):
                    best = candidate
                    improved = True
        route = best
    return best


def solve_heuristic(cost: np.ndarray) -> SolutionResult:
    start = time.perf_counter()
    initial = nearest_neighbor_route(cost)
    improved = two_opt(initial, cost)
    total = route_cost(improved, cost)
    runtime = time.perf_counter() - start
    return SolutionResult("Heuristic-NN-2opt", improved, total, runtime)


def summarize_solution(result: SolutionResult) -> str:
    route_str = " -> ".join(str(i) for i in result.route)
    return (
        f"Method: {result.method}\n"
        f"Route: {route_str}\n"
        f"Total predicted time (minutes): {result.total_minutes:.2f}\n"
        f"Runtime (seconds): {result.runtime_seconds:.4f}\n"
    )


### GENETIC ALGORITHM (Heuristic Method)

In [26]:
class GeneticTSP:
    def __init__(
        self,
        C: np.ndarray,
        pop_size: int     = 150,
        n_gen: int        = 800,
        cx_prob: float    = 0.85,
        mut_prob: float   = 0.15,
        tournament_k: int = 5,
        elite_frac: float = 0.10,
        two_opt_freq: int = 50,
        seed: int         = 42,
        time_limit: float = 60.0,
    ):
        self.C           = C
        self.n           = C.shape[0] - 1
        self.pop_size    = pop_size
        self.n_gen       = n_gen
        self.cx_prob     = cx_prob
        self.mut_prob    = mut_prob
        self.k           = tournament_k
        self.n_elite     = max(2, int(elite_frac * pop_size))
        self.two_opt_freq = two_opt_freq
        self.rng         = np.random.default_rng(seed)
        self.time_limit  = time_limit

    def _fitness(self, chrom: np.ndarray) -> float:
        return (
            self.C[0, chrom[0]]
            + float(np.sum(self.C[chrom[:-1], chrom[1:]]))
            + self.C[chrom[-1], 0]
        )

    def _greedy_tour(self) -> np.ndarray:
        unvisited = list(range(1, self.n + 1))
        tour, cur = [], 0
        while unvisited:
            nxt = min(unvisited, key=lambda j: self.C[cur, j])
            tour.append(nxt); unvisited.remove(nxt); cur = nxt
        return np.array(tour)

    def _init_population(self) -> List[np.ndarray]:
        nodes = np.arange(1, self.n + 1)
        pop   = [self.rng.permutation(nodes) for _ in range(self.pop_size)]
        pop[0] = self._greedy_tour()   # seed one good individual
        return pop

    def _tournament(self, pop: List[np.ndarray], fits: np.ndarray) -> np.ndarray:
        idx  = self.rng.choice(len(pop), self.k, replace=False)
        best = idx[int(np.argmin(fits[idx]))]
        return pop[best].copy()

    def _ox1(self, p1: np.ndarray, p2: np.ndarray) -> np.ndarray:
        n     = len(p1)
        a, b  = sorted(self.rng.choice(n, 2, replace=False))
        child = np.full(n, -1, dtype=int)
        child[a:b] = p1[a:b]
        fill  = [g for g in p2 if g not in child]
        ptr   = 0
        for i in range(n):
            if child[i] == -1:
                child[i] = fill[ptr]; ptr += 1
        return child

    def _swap_mutate(self, chrom: np.ndarray) -> np.ndarray:
        i, j       = self.rng.choice(len(chrom), 2, replace=False)
        chrom[i], chrom[j] = chrom[j], chrom[i]
        return chrom

    def _two_opt(self, chrom: np.ndarray) -> np.ndarray:
        best, best_cost = chrom.copy(), self._fitness(chrom)
        improved = True
        while improved:
            improved = False
            for i in range(len(best) - 1):
                for j in range(i + 2, len(best)):
                    cand         = best.copy()
                    cand[i:j+1]  = best[i:j+1][::-1]
                    c = self._fitness(cand)
                    if c < best_cost - 1e-9:
                        best, best_cost = cand, c
                        improved = True
        return best

    def run(self) -> dict:
        t0   = time.perf_counter()
        pop  = self._init_population()
        fits = np.array([self._fitness(c) for c in pop])

        best_cost  = float(np.min(fits))
        best_chrom = pop[int(np.argmin(fits))].copy()
        history    = [best_cost]

        print(f"  GA gen {'0':>4} | best cost = {best_cost:.4f} km  "
              f"(pop={self.pop_size}, seeded with greedy tour)")

        for gen in range(1, self.n_gen + 1):
            if time.perf_counter() - t0 > self.time_limit:
                print(f"  GA stopped at gen {gen} – time limit reached.")
                break

            elite_idx = np.argsort(fits)[: self.n_elite]
            new_pop   = [pop[i].copy() for i in elite_idx]

            if gen % self.two_opt_freq == 0:
                new_pop[0]          = self._two_opt(new_pop[0])
                fits[elite_idx[0]]  = self._fitness(new_pop[0])

            while len(new_pop) < self.pop_size:
                p1    = self._tournament(pop, fits)
                p2    = self._tournament(pop, fits)
                child = self._ox1(p1, p2) if self.rng.random() < self.cx_prob else p1.copy()
                if self.rng.random() < self.mut_prob:
                    child = self._swap_mutate(child)
                new_pop.append(child)

            pop  = new_pop
            fits = np.array([self._fitness(c) for c in pop])

            gen_best = float(np.min(fits))
            if gen_best < best_cost:
                best_cost  = gen_best
                best_chrom = pop[int(np.argmin(fits))].copy()
            history.append(best_cost)

            if gen % 100 == 0:
                print(f"  GA gen {gen:>4} | best cost = {best_cost:.4f} km")

        elapsed = time.perf_counter() - t0
        return {
            "solver":      "Genetic Algorithm (OX1 + 2-opt)",
            "route":       best_chrom.tolist(),
            "cost":        best_cost,
            "runtime_s":   elapsed,
            "feasible":    True,
            "status":      "Near-optimal heuristic",
            "convergence": history,
        }


### PRINT RESULTS (Exact VS Heuristic)

In [27]:
def print_report(results: list, orders: pd.DataFrame) -> None:
    print("\n" + "=" * 72)
    print("DELIVERY OPTIMIZATION – RESULTS COMPARISON")
    print("=" * 72)
    print(f"{'Solver':<38} {'Cost(km)':>10} {'Runtime(s)':>11} {'Feasible':>9}  Status")
    print("-" * 72)
    for r in results:
        cost_s = f"{r['cost']:.4f}" if r["cost"] < 1e15 else "N/A"
        print(
            f"{r['solver']:<38} {cost_s:>10} {r['runtime_s']:>11.4f}"
            f" {'Yes' if r['feasible'] else 'No':>9}  {r['status']}"
        )
    print("=" * 72)

    # --- Intra-Category Comparison ---
    exact_methods = [r for r in results if "Exact" in r["solver"] or "Held-Karp" in r["solver"]]
    heuristic_methods = [r for r in results if "Heuristic" in r["solver"] or "Genetic" in r["solver"] or "Local Search" in r["solver"]]

    print("\n── Category Analysis ──")

    if exact_methods:
        best_exact = min(exact_methods, key=lambda x: (x['cost'], x['runtime_s']))
        print(f"BEST EXACT METHOD    : {best_exact['solver']} (Time: {best_exact['runtime_s']:.4f}s)")
        print("  Note: Exact methods always find the optimal cost; efficiency is the tie-breaker.")

    if heuristic_methods:
        best_heur = min(heuristic_methods, key=lambda x: (x['cost'], x['runtime_s']))
        print(f"BEST HEURISTIC METHOD: {best_heur['solver']} (Cost: {best_heur['cost']:.4f} km)")
        print("  Note: Heuristics prioritize speed and scalability over guaranteed optimality.")

    print("\n── Scalability & Complexity ──")
    print("  Exact Methods (ILP/DP): Best for small batches (n < 20). Guaranteed optimal.")
    print("  Heuristics (GA/Search): Best for large-scale operations. Very fast.")

In [28]:
inputObj = {
    "data": "amazon_delivery.csv",
    "orders": 14,
    "seed": 42,
    "time_limit": 120
}

# 1. Data Preparation
depot, orders = load_and_prepare_data(inputObj["data"], inputObj["orders"], inputObj["seed"])
cost_matrix = build_cost_matrix(depot, orders)

print("=== Unified Delivery Optimization Comparison ===")
print(f"Depot: ({depot[0]:.6f}, {depot[1]:.6f})")
print(f"Orders optimized: {len(orders)}\n")

# --- Execution ---
exact_ilp = solve_exact_tsp_mtz(cost_matrix, time_limit=inputObj["time_limit"])
heuristic_ls = solve_heuristic(cost_matrix)
hk_res = solve_held_karp(cost_matrix)
ga_instance = GeneticTSP(cost_matrix, seed=inputObj["seed"], time_limit=float(inputObj["time_limit"]))
ga_res = ga_instance.run()

# --- Structuring Results ---
results_list = [
    {
        "solver": "Exact-MTZ (ILP)",
        "route": exact_ilp.route[1:-1],
        "cost": exact_ilp.total_minutes,
        "runtime_s": exact_ilp.runtime_seconds,
        "feasible": exact_ilp.total_minutes < 1e15,
        "status": "Optimal"
    },
    {
        "solver": "Local Search (NN-2opt) (Heuristic)",
        "route": heuristic_ls.route[1:-1],
        "cost": heuristic_ls.total_minutes,
        "runtime_s": heuristic_ls.runtime_seconds,
        "feasible": True,
        "status": "Heuristic"
    },
    hk_res,
    ga_res
]

print_report(results_list, orders)

=== Unified Delivery Optimization Comparison ===
Depot: (0.000000, 0.000000)
Orders optimized: 14

  GA gen    0 | best cost = 136.6841 km  (pop=150, seeded with greedy tour)
  GA gen  100 | best cost = 136.6841 km
  GA gen  200 | best cost = 136.6841 km
  GA gen  300 | best cost = 136.6841 km
  GA gen  400 | best cost = 136.6841 km
  GA gen  500 | best cost = 136.6841 km
  GA gen  600 | best cost = 136.6841 km
  GA gen  700 | best cost = 136.6841 km
  GA gen  800 | best cost = 136.6841 km

DELIVERY OPTIMIZATION – RESULTS COMPARISON
Solver                                   Cost(km)  Runtime(s)  Feasible  Status
------------------------------------------------------------------------
Exact-MTZ (ILP)                          136.6841    116.2710       Yes  Optimal
Local Search (NN-2opt) (Heuristic)       136.6841      0.0027       Yes  Heuristic
Held-Karp DP (Exact)                     136.6841      4.5856       Yes  Globally optimal
Genetic Algorithm (OX1 + 2-opt)          136.6841     

In [29]:
inputObj = {
    "data": "amazon_delivery.csv",
    "orders": 14,
    "seed": 42,
    "time_limit": 120
}

# load & preprocess
depot, orders = load_and_prepare_data(inputObj["data"], inputObj["orders"], inputObj["seed"])

# build cost matrix
print("[Building cost matrix…]")
C = build_cost_matrix(depot, orders)
n = len(orders)
print(f"[Cost matrix] {n+1}×{n+1}  (depot + {n} deliveries)\n")

results = []

# ── SOLVER 1: Held-Karp DP ────────────────────────────────────
if n > 20:
    print(f"[1/2] WARNING: n={n} > 20; Held-Karp O(n²·2ⁿ) may be very slow. "
            "Held-Karp DP skipped.")

print("[1/2] Running Held-Karp Dynamic Programming (exact)…")
dp_res = solve_held_karp(C)
print(f"      Done.  Cost={dp_res['cost']:.4f} km  "
        f"Time={dp_res['runtime_s']:.4f}s  Status={dp_res['status']}")
results.append(dp_res)

# ── SOLVER 2: Genetic Algorithm ──────────────────────────────
print(f"\n[2/2] Running Genetic Algorithm (time limit={inputObj['time_limit']}s)…")
ga = GeneticTSP(
    C,
    pop_size      = 150,
    n_gen         = 800,
    cx_prob       = 0.85,
    mut_prob      = 0.15,
    tournament_k  = 5,
    elite_frac    = 0.10,
    two_opt_freq  = 50,
    seed          = inputObj["seed"],
    time_limit    = float(inputObj["time_limit"]),
)
ga_res = ga.run()
print(f"\n      Done.  Cost={ga_res['cost']:.4f} km  "
        f"Time={ga_res['runtime_s']:.4f}s  Status={ga_res['status']}")
results.append(ga_res)

print_report(results, orders)

[Building cost matrix…]
[Cost matrix] 15×15  (depot + 14 deliveries)

[1/2] Running Held-Karp Dynamic Programming (exact)…
      Done.  Cost=136.6841 km  Time=2.5975s  Status=Globally optimal

[2/2] Running Genetic Algorithm (time limit=120s)…
  GA gen    0 | best cost = 136.6841 km  (pop=150, seeded with greedy tour)
  GA gen  100 | best cost = 136.6841 km
  GA gen  200 | best cost = 136.6841 km
  GA gen  300 | best cost = 136.6841 km
  GA gen  400 | best cost = 136.6841 km
  GA gen  500 | best cost = 136.6841 km
  GA gen  600 | best cost = 136.6841 km
  GA gen  700 | best cost = 136.6841 km
  GA gen  800 | best cost = 136.6841 km

      Done.  Cost=136.6841 km  Time=78.6404s  Status=Near-optimal heuristic

DELIVERY OPTIMIZATION – RESULTS COMPARISON
Solver                                   Cost(km)  Runtime(s)  Feasible  Status
------------------------------------------------------------------------
Held-Karp DP (Exact)                     136.6841      2.5975       Yes  Globally opti